In [ ]:
# ==================================================
# TRAM TRACK SAFETY SYSTEM (FINAL – YOLO ALWAYS ON 🔒)
# ==================================================

import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from ultralytics import YOLO

# =========================
# PATHS
# =========================
VIDEO_PATH = r"C:\akbar\AKBARPK\AKBAR\PROJECTS\AAA_TRAM MONITORING SYSYTEM\FINAL STRUCTURE\testing video\Adobe Express 2026-01-22 09.52.42.mp4"

YOLO_GENERAL_PATH = r"C:\akbar\AKBARPK\AKBAR\PROJECTS\AAA_TRAM MONITORING SYSYTEM\FINAL STRUCTURE\yolo_large_model.pt"
YOLO_CUSTOM_PATH  = r"C:\akbar\AKBARPK\AKBAR\PROJECTS\AAA_TRAM MONITORING SYSYTEM\FINAL STRUCTURE\yolo_wheel_stone.pt"

UNET_MODEL_PATH = r"C:\akbar\AKBARPK\AKBAR\PROJECTS\AAA_TRAM MONITORING SYSYTEM\FINAL STRUCTURE\track_segmentation_unet_final_ttv.h5"

OUTPUT_VIDEO_PATH = r"C:\akbar\AKBARPK\AKBAR\PROJECTS\AAA_TRAM MONITORING SYSYTEM\FINAL STRUCTURE\FINAL_OUTPUT\FINAL.mp4"

# =========================
# PARAMETERS
# =========================
IMG_SIZE = (192, 192)
THRESHOLD = 0.5

YOLO_CONF = 0.25
RED_RATIO = 0.02
YELLOW_RATIO = 0.005

RED_ALPHA = 0.30       # 🔴 transparency
YELLOW_ALPHA = 0.25    # 🟡 transparency

# =========================
# ALLOWED CLASSES
# =========================
GENERAL_ALLOWED_IDS = {
    0, 1, 2, 3, 5, 7,
    13,
    14, 15, 16, 17, 18, 19,
    20, 21, 22, 23,
    56, 57
}
CUSTOM_ALLOWED_IDS = {0, 1}

# =========================
# LOAD MODELS
# =========================
yolo_custom = YOLO(YOLO_CUSTOM_PATH)
yolo_general = YOLO(YOLO_GENERAL_PATH)
unet = load_model(UNET_MODEL_PATH)

print("✅ Models loaded")

# =========================
# VIDEO SETUP
# =========================
cap = cv2.VideoCapture(VIDEO_PATH)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)

os.makedirs(os.path.dirname(OUTPUT_VIDEO_PATH), exist_ok=True)
writer = cv2.VideoWriter(
    OUTPUT_VIDEO_PATH,
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (w, h)
)

# =========================
# SEGMENTATION
# =========================
def get_track_zones(frame):
    img = cv2.resize(frame, IMG_SIZE).astype(np.float32) / 255.0
    pred = unet.predict(img[np.newaxis, ...], verbose=0)[0, :, :, 0]

    mask = (pred > THRESHOLD).astype(np.uint8)
    mask = cv2.resize(mask, (w, h), interpolation=cv2.INTER_NEAREST)

    kernel = np.ones((35, 35), np.uint8)
    yellow_zone = cv2.dilate(mask, kernel)

    return mask, yellow_zone

# =========================
# PROCESS VIDEO
# =========================
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    overlay = frame.copy()
    danger_red = False
    danger_yellow = False

    # -------- SEGMENTATION --------
    red_zone, yellow_zone = get_track_zones(frame)

    # ---- TRANSPARENT RED ZONE ----
    red_mask = np.zeros_like(frame)
    red_mask[red_zone == 1] = (0, 0, 255)
    overlay = cv2.addWeighted(overlay, 1.0, red_mask, RED_ALPHA, 0)

    # ---- TRANSPARENT YELLOW ZONE ----
    yellow_mask = np.zeros_like(frame)
    yellow_mask[(yellow_zone == 1) & (red_zone == 0)] = (0, 255, 255)
    overlay = cv2.addWeighted(overlay, 1.0, yellow_mask, YELLOW_ALPHA, 0)

    # =========================
    # YOLO CUSTOM (FIRST)
    # =========================
    results = yolo_custom.predict(frame, conf=YOLO_CONF, verbose=False)

    for r in results:
        if r.boxes is None:
            continue

        for box in r.boxes:
            cls_id = int(box.cls[0])
            if cls_id not in CUSTOM_ALLOWED_IDS:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            area = max((x2 - x1) * (y2 - y1), 1)

            red_overlap = np.sum(red_zone[y1:y2, x1:x2]) / area
            yellow_overlap = np.sum(yellow_zone[y1:y2, x1:x2]) / area

            label = yolo_custom.names[cls_id]

            if red_overlap > RED_RATIO:
                color = (0, 0, 255)
                text = "CRITICAL"
                danger_red = True
            elif yellow_overlap > YELLOW_RATIO:
                color = (0, 255, 255)
                text = "WARNING"
                danger_yellow = True
            else:
                color = (255, 0, 0)
                text = "DETECTED"

            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
            cv2.putText(overlay, f"{label} {text}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # =========================
    # YOLO GENERAL (SECOND)
    # =========================
    results = yolo_general.predict(frame, conf=YOLO_CONF, verbose=False)

    for r in results:
        if r.boxes is None:
            continue

        for box in r.boxes:
            cls_id = int(box.cls[0])
            if cls_id not in GENERAL_ALLOWED_IDS:
                continue

            x1, y1, x2, y2 = map(int, box.xyxy[0])
            area = max((x2 - x1) * (y2 - y1), 1)

            red_overlap = np.sum(red_zone[y1:y2, x1:x2]) / area
            yellow_overlap = np.sum(yellow_zone[y1:y2, x1:x2]) / area

            label = yolo_general.names[cls_id]

            if red_overlap > RED_RATIO:
                color = (0, 0, 255)
                text = "ON TRACK"
                danger_red = True
            elif yellow_overlap > YELLOW_RATIO:
                color = (0, 255, 255)
                text = "NEAR TRACK"
                danger_yellow = True
            else:
                color = (0, 255, 0)
                text = "SAFE"

            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, 2)
            cv2.putText(overlay, f"{label} {text}",
                        (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    # -------- GLOBAL ALERT --------
    if danger_red:
        msg = "⚠ CRITICAL: OBJECT ON TRACK ⚠"
        color = (0, 0, 255)
    elif danger_yellow:
        msg = "⚠ WARNING: OBJECT NEAR TRACK ⚠"
        color = (0, 255, 255)
    else:
        msg = None

    if msg:
        cv2.putText(overlay, msg, (40, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 3)

    writer.write(overlay)
    cv2.imshow("Tram Track Safety System", overlay)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# =========================
# CLEANUP
# =========================
cap.release()
writer.release()
cv2.destroyAllWindows()

print("✅ FINAL OUTPUT SAVED TO:", OUTPUT_VIDEO_PATH)
